In [1]:
# --- Notebook Test Engine: injected setup (auto-generated) ---
import os as _os
for _k in ('AWS_ACCESS_KEY_ID','AWS_SECRET_ACCESS_KEY','AWS_SESSION_TOKEN','AWS_CREDENTIAL_EXPIRATION'):
    _os.environ.pop(_k, None)
_os.environ.setdefault('AWS_DEFAULT_REGION', 'us-west-2')
_os.environ.setdefault('AWS_REGION', 'us-west-2')
_nte_role = 'arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole'
try:
    from sagemaker.core.helper import session_helper as _sh
    _sh.get_execution_role = lambda *a, **k: _nte_role
except Exception:
    pass
try:
    _nte_cf = _os.environ.get('AWS_SHARED_CREDENTIALS_FILE')
    if _nte_cf and _os.path.exists(_nte_cf):
        import configparser as _cfp, datetime as _dt
        import boto3 as _b3, botocore.session as _bcs
        from botocore.credentials import RefreshableCredentials as _RC
        _nte_prof = _os.environ.get('AWS_PROFILE', 'default')
        def _nte_load():
            _cp = _cfp.ConfigParser(); _cp.read(_nte_cf)
            _s = _cp[_nte_prof]
            _tok = _s.get('aws_session_token')
            if not _tok:
                raise RuntimeError('nte: no session token in creds file')
            return {'access_key': _s['aws_access_key_id'],
                    'secret_key': _s['aws_secret_access_key'],
                    'token': _tok,
                    'expiry_time': (_dt.datetime.now(_dt.timezone.utc)
                                    + _dt.timedelta(minutes=9)).isoformat()}
        _nte_creds = _RC.create_from_metadata(metadata=_nte_load(),
                                              refresh_using=_nte_load,
                                              method='nte-shared-file')
        def _nte_get_credentials(self, *a, **k):
            # Honor sessions that were handed explicit credentials
            # (e.g. notebooks that assume_role into other accounts):
            # clobbering them would silently run every cross-account
            # client as the ambient identity. Bare/SDK-created sessions
            # (no explicit creds) still get the refreshable shared-file
            # creds so long-running waits survive credential rotation.
            _ex = getattr(self, '_credentials', None)
            if _ex is not None:
                return _ex
            return _nte_creds
        _bcs.Session.get_credentials = _nte_get_credentials
        _b3.setup_default_session(botocore_session=_bcs.get_session())
        print('nte: refreshable shared-file creds installed')
except Exception as _e:
    print('nte: refreshable-creds shim skipped:', _e)
try:
    from sagemaker.core.resources import ModelPackageGroup as _NteMPG
    _nte_mpg_create = _NteMPG.create
    def _nte_mpg_get_or_create(*_a, **_k):
        try:
            return _nte_mpg_create(*_a, **_k)
        except Exception as _e2:
            if 'already exists' in str(_e2).lower():
                _name = _k.get('model_package_group_name') or (_a[0] if _a else None)
                return _NteMPG.get(_name)
            raise
    _NteMPG.create = staticmethod(_nte_mpg_get_or_create)
    print('nte: ModelPackageGroup.create is now get-or-create')
except Exception as _e:
    print('nte: ModelPackageGroup idempotency shim skipped:', _e)


nte: ModelPackageGroup.create is now get-or-create


# SageMaker V3 Local Container Mode Example

This notebook demonstrates how to use SageMaker V3 ModelBuilder in Local Container mode for testing models in Docker containers locally.

In [2]:
# Import required libraries
import json
import uuid
import tempfile
import os
import shutil
import torch
import torch.nn as nn

from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.serve.spec.inference_spec import InferenceSpec
from sagemaker.serve.builder.schema_builder import SchemaBuilder
from sagemaker.serve.utils.types import ModelServer
from sagemaker.serve.mode.function_pointers import Mode

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Fetched defaults config from location: /tmp/sm_config_930.yaml


In [3]:
# NOTE: Local mode requires Docker to be installed and running. 
# If Docker is not in your system PATH, you may need to define the Docker path in one of the top cells.
# Here is an example:
import os
os.environ['PATH'] = '/usr/local/bin:/Applications/Docker.app/Contents/Resources/bin:' + os.environ['PATH']

## Step 1: Create a PyTorch Model

Create and save a simple PyTorch model for local container testing.

In [4]:
class SimpleModel(nn.Module):
    """Simple PyTorch model for testing."""
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(4, 2)
    
    def forward(self, x):
        return torch.softmax(self.linear(x), dim=1)

def save_pytorch_model(model_path: str):
    """Save PyTorch model for testing."""
    model = SimpleModel()
    sample_input = torch.tensor([[1.0, 2.0, 3.0, 4.0]], dtype=torch.float32)
    traced_model = torch.jit.trace(model, sample_input)
    model_file = os.path.join(model_path, "model.pt")
    torch.jit.save(traced_model, model_file)
    return model_file

# Create temporary model directory and save model
temp_model_path = tempfile.mkdtemp()
model_file = save_pytorch_model(temp_model_path)
print(f"Model saved to: {model_file}")

Model saved to: /tmp/tmpyw2mfxpx/model.pt


## Step 2: Define PyTorch InferenceSpec

Create an InferenceSpec that can load and run our PyTorch model.

In [5]:
class PyTorchInferenceSpec(InferenceSpec):
    """PyTorch InferenceSpec for local container mode."""
    
    def __init__(self, model_path=None):
        self.model_path = model_path
    
    def prepare(self, model_dir: str):
        """Prepare PyTorch model artifacts."""
        if self.model_path:
            src_model = os.path.join(self.model_path, "model.pt")
            dst_model = os.path.join(model_dir, "model.pt")
            if os.path.exists(src_model) and src_model != dst_model:
                shutil.copy2(src_model, dst_model)
    
    def load(self, model_dir: str):
        """Load PyTorch model."""
        model_path = os.path.join(model_dir, "model.pt")
        
        if os.path.exists(model_path):
            model = torch.jit.load(model_path, map_location='cpu')
        else:
            model = SimpleModel()
        
        model.eval()
        return model
    
    def invoke(self, input_object, model):
        """PyTorch inference."""
        if isinstance(input_object, dict) and "data" in input_object:
            input_data = input_object["data"]
        else:
            input_data = input_object
        
        if isinstance(input_data, list):
            input_tensor = torch.tensor(input_data, dtype=torch.float32)
        else:
            input_tensor = torch.tensor(input_data.tolist() if hasattr(input_data, 'tolist') else input_data, dtype=torch.float32)
        
        with torch.no_grad():
            output = model(input_tensor)
            return output.tolist()

print("PyTorch InferenceSpec defined successfully!")

PyTorch InferenceSpec defined successfully!


## Step 3: Create Schema Builder

Define the input/output schema for our PyTorch model.

In [6]:
# Create PyTorch schema builder
sample_input = [[1.0, 2.0, 3.0, 4.0]]
sample_output = [[0.6, 0.4]]
schema_builder = SchemaBuilder(sample_input, sample_output)

print("Schema builder created successfully!")

Schema builder created successfully!


## Step 4: Configure ModelBuilder for Local Container Mode

Set up ModelBuilder to run in LOCAL_CONTAINER mode with Docker.

In [7]:
# Configuration
MODEL_NAME_PREFIX = "pytorch-local"
ENDPOINT_NAME_PREFIX = "pytorch-local"

# Generate unique identifiers
unique_id = str(uuid.uuid4())[:8]
model_name = f"{MODEL_NAME_PREFIX}-{unique_id}"
endpoint_name = f"{ENDPOINT_NAME_PREFIX}-{unique_id}"

# Create ModelBuilder in LOCAL_CONTAINER mode
inference_spec = PyTorchInferenceSpec(model_path=temp_model_path)
model_builder = ModelBuilder(image_uri="763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-inference:2.6.0-cpu-py312", 
    inference_spec=inference_spec,
    model_server=ModelServer.TORCHSERVE,
    schema_builder=schema_builder,
    mode=Mode.LOCAL_CONTAINER  # This enables Docker container mode
)

print(f"ModelBuilder configured for local container model: {model_name}")
print(f"Target endpoint: {endpoint_name}")
print("Note: This will use Docker containers locally!")

[07/20/26 07:19:03] DEBUG    Auto-detecting optimal instance type for model...           ]8;id=10714776;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=10714777;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#341\341]8;;\

                    DEBUG    Using default CPU instance type: ml.m5.large                ]8;id=10714783;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=10714784;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#375\375]8;;\

[07/20/26 07:19:04] INFO     Cannot simulate policies for                                  ]8;id=10714791;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=10714792;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-CodeBuildR                         
                             ole' (access denied); permission verdict unknown.                                     

                    WARNING  Could not verify permissions for role                         ]8;id=10714798;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=10714799;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-CodeBuildR                         
                             ole' (caller lacks iam:SimulatePrincipalPolicy). Proceeding                           
                             with it. If the operation later fails with an access-denied                           
                             error, ensure the role has the required permissions for                               
                             'serving' (see                                                                        
                             IamRoleResolver().get_required_actions('serving')) or create                          
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='serving').                         

ModelBuilder configured for local container model: pytorch-local-f05361e1
Target endpoint: pytorch-local-f05361e1
Note: This will use Docker containers locally!


## Step 5: Build the Model

Build the model artifacts for containerized deployment.

In [8]:
# Build the model
local_model = model_builder.build(model_name=model_name)
print(f"Model Successfully Created: {local_model.model_name}")

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=10714806;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=10714807;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/20/26 07:19:05] DEBUG    Either inference spec or model is provided. ModelBuilder   ]8;id=10714813;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=10714814;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#1380\1380]8;;\
                             is not handling MLflow model input                                                    

                    DEBUG    Skipping auto-detection as image_uri is provided:           ]8;id=10714820;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=10714821;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#899\899]8;;\
                             763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-infere                           
                             nce:2.6.0-cpu-py312                                                                   

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /root/.config/sagemaker/config.yaml


Scanning for dependencies:   0%|                                            | 0/228 [00:00<?, ?it/s]

...

[07/20/26 07:19:59] INFO     Cannot simulate policies for                                  ]8;id=10714826;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=10714827;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-CodeBuildR                         
                             ole' (access denied); permission verdict unknown.                                     

                    WARNING  Could not verify permissions for role                         ]8;id=10714832;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=10714833;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-CodeBuildR                         
                             ole' (caller lacks iam:SimulatePrincipalPolicy). Proceeding                           
                             with it. If the operation later fails with an access-denied                           
                             error, ensure the role has the required permissions for                               
                             'serving' (see                                                                        
                             IamRoleResolver().get_required_actions('serving')) or create                          
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='serving').                         

[07/20/26 07:20:00] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=10714838;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=10714839;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     ✅ Model has been created: 'pytorch-local-f05361e1' using server ]8;id=10714846;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=10714847;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/serve/model_builder.py#3656\3656]8;;\
                             TORCHSERVE in LOCAL_CONTAINER mode                                                    

Model Successfully Created: pytorch-local-f05361e1


## Step 6: Deploy in Local Container

Deploy the model in a local Docker container. This may take a few minutes to pull the container image and ping the container until it is live. This is a normal part of the deployment process.

In [9]:
# Deploy locally in container mode
print("Starting local container deployment...")
print("Note: This may take a few minutes to pull the Docker image on first run.")

local_endpoint = model_builder.deploy_local(
    endpoint_name=endpoint_name,
    wait=True,
    container_timeout_in_seconds=1200  # 20 minutes timeout
)

print(f"Local Container Endpoint Successfully Created: {endpoint_name}")
print("Container is now running and ready for inference!")

Starting local container deployment...
Note: This may take a few minutes to pull the Docker image on first run.


                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=10714852;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=10714853;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=10714858;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=10714859;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/20/26 07:20:01] INFO     Successfully authenticated with ECR                        ]8;id=10714866;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/serve/mode/local_container_mode.py\local_container_mode.py]8;;\:]8;id=10714867;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/serve/mode/local_container_mode.py#259\259]8;;\

                    INFO     Pulling image                                              ]8;id=10714873;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/serve/mode/local_container_mode.py\local_container_mode.py]8;;\:]8;id=10714874;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/serve/mode/local_container_mode.py#272\272]8;;\
                             763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-infer                            
                             ence:2.6.0-cpu-py312 from repository...                                               

...

[07/20/26 07:23:11] WARNING  Using the short-lived AWS credentials found in session. They might       ]8;id=10714972;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/local/image.py\image.py]8;;\:]8;id=10714973;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/local/image.py#1138\1138]8;;\
                             expire while running.                                                                 

                    INFO     docker compose file:                                                      ]8;id=10714979;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/local/image.py\image.py]8;;\:]8;id=10714980;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/local/image.py#787\787]8;;\
                             networks:                                                                             
                               sagemaker-local:                                                                    
                                 name: sagemaker-local                                                             
                             services:                                                                             
                               algo-1-lmu9v:                                                                       
                                 command: serve                                                                    
                                 container_name: 7snqr8ltrc-algo-1-lmu9v                                           
                                 environment:                                                                      
                                 - '[Masked]'                                                                      
                                 - '[Masked]'                                                                      
                                 - '[Masked]'                                                                      
                                 - '[Masked]'                                                                      
                                 - '[Masked]'                                                                      
                                 - '[Masked]'                                                                      
                                 - '[Masked]'                                                                      
                                 image:                                                                            
                             763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-inference:2.6.0-cpu-             
                             py312                                                                                 
                                 networks:                                                                         
                                   sagemaker-local:                                                                
                                     aliases:                                                                      
                                     - algo-1-lmu9v                                                                
                                 ports:                                                                            
                                 - 8080:8080                                                                       
                                 stdin_open: true                                                                  
                                 tty: true                                                                         
                                 volumes:                                                                          
                                 -                                                                                 
                             /tmp/sagemaker/model-builder/4920783c840b11f19b580242ac110002:/opt/ml/mod             
                             el                                                                                    
                             version: '2.3'                                                                        
                                            

                    INFO     docker command: docker compose -f /tmp/tmp96fdm7d7/docker-compose.yaml up ]8;id=10714986;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/local/image.py\image.py]8;;\:]8;id=10714987;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/local/image.py#811\811]8;;\
                             --build --abort-on-container-exit                                                     

                    INFO     Checking if serving container is up, attempt: 5                        ]8;id=10714994;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/local/entities.py\entities.py]8;;\:]8;id=10714995;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/local/entities.py#671\671]8;;\

Local Container Endpoint Successfully Created: pytorch-local-f05361e1
Container is now running and ready for inference!


## Step 7: Test the Containerized Model

Send test requests to the model running in the local container.

In [10]:
# Test 1: Single prediction
test_input_1 = [[1.0, 2.0, 3.0, 4.0]]

response_1 = local_endpoint.invoke(
    body=json.dumps(test_input_1),
    content_type="application/json"
)

response_data_1 = response_1.body.read().decode('utf-8')
parsed_response_1 = json.loads(response_data_1)
print(f"Test 1 - Single prediction: {parsed_response_1}")

Test 1 - Single prediction: [[0.7868615388870239, 0.21313847601413727]]


In [11]:
# Test 2: Batch prediction
test_input_2 = [
    [1.0, 2.0, 3.0, 4.0],
    [0.5, 1.5, 2.5, 3.5],
    [2.0, 3.0, 4.0, 5.0]
]

response_2 = local_endpoint.invoke(
    body=json.dumps(test_input_2),
    content_type="application/json"
)

response_data_2 = response_2.body.read().decode('utf-8')
parsed_response_2 = json.loads(response_data_2)
print(f"Test 2 - Batch prediction: {parsed_response_2}")

Test 2 - Batch prediction: [[0.7868615388870239, 0.21313847601413727], [0.7003034353256226, 0.29969653487205505], [0.9021058082580566, 0.09789419919252396]]


In [12]:
# Test 3: Edge case - different input ranges
test_input_3 = [[0.1, 0.2, 0.3, 0.4]]

response_3 = local_endpoint.invoke(
    body=json.dumps(test_input_3),
    content_type="application/json"
)

response_data_3 = response_3.body.read().decode('utf-8')
parsed_response_3 = json.loads(response_data_3)
print(f"Test 3 - Edge case: {parsed_response_3}")

Test 3 - Edge case: [[0.32905688881874084, 0.6709431409835815]]


## Step 8: Container Information

Get information about the running container.

In [13]:
# Display container information
print("Container Information:")
print(f"- Endpoint Name: {local_endpoint.endpoint_name}")
print(f"- Model Server: TorchServe")
print(f"- Container Mode: LOCAL_CONTAINER")
print(f"- Model Path: {temp_model_path}")

# You can also check Docker containers running
print("\nTo see the running container, you can run:")
print("docker ps")

Container Information:
- Endpoint Name: pytorch-local-f05361e1
- Model Server: TorchServe
- Container Mode: LOCAL_CONTAINER
- Model Path: /tmp/tmpyw2mfxpx

To see the running container, you can run:
docker ps


## Step 9: Clean Up

Clean up the local container and temporary files.

In [14]:
# Clean up temporary model files
shutil.rmtree(temp_model_path)
print("Temporary model files cleaned up!")

# Note: Local container will be automatically cleaned up when the process ends
print("Local container will be automatically stopped when this notebook session ends.")
print("No AWS resources were created, so no cloud cleanup needed.")

Temporary model files cleaned up!
Local container will be automatically stopped when this notebook session ends.
No AWS resources were created, so no cloud cleanup needed.


## Summary

This notebook demonstrated:
1. Creating and saving a PyTorch model
2. Defining a PyTorch InferenceSpec with prepare(), load(), and invoke() methods
3. Configuring ModelBuilder for LOCAL_CONTAINER mode
4. Building and deploying models in local Docker containers
5. Testing containerized models with various inputs
6. Proper cleanup of local resources

## Benefits of Local Container Mode:
- **Container parity**: Same environment as SageMaker endpoints
- **No AWS costs**: Runs entirely locally
- **Realistic testing**: Uses actual model serving containers
- **Debugging friendly**: Can inspect container logs and behavior
- **Dependency isolation**: Container handles all dependencies

## When to Use Local Container Mode:
- Testing models before deploying to SageMaker
- Debugging inference issues
- Validating custom inference code
- Development with realistic serving environment
- CI/CD pipeline testing

Local container mode provides the perfect balance between local development speed and production environment fidelity!